### Support Vector machaine is a supervised machine learning algorithm that can be used for both classification and regression tasks. 

#### Many lines can separate  classes, but SVC chooses the best one.

#### The best hyperplane is the one that maximizes the margin (distance between the hyperplane and the nearest data points of each class).

#### The points closest to the hyperplane are called Support Vectors.


#### w is the weight vector — it defines the orientation of the hyperplane and points in the direction perpendicular to it. The decision function is:


#### f(x)=w⋅x+b (decision boundary) 

#### if f(x)>0, then class 1

#### if f(x)<0, then class 2

#### The bias term b shifts the hyperplane away from the origin and allows for better separation of the classes.

### distance from a point x to the hyperplane :
#### distance = |f(x)| / ||w||, where ||w|| is the norm of the weight vector w.


#### margin = 2/||w||, the distance between the support vectors of the two classes. (maximization  of margin leads to lower w )



### Hinge loss function 
#### Hinge loss for a classified data point (x,y) hinge loss = max(0, 1 - y * f(x)) = 0  (correctly classified)

#### in bwtween two margin line , if a data point exists then hinge loss = 1 - y * f(x) =0.5 (also correctly calssified)

#### if a data point is misclassified then hinge loss = 1 - y * f(x) = 1.5 (misclassified)

### Hard Margin (for linearly separable data)

#### In hard margin SVM, all data points must be correctly classified with no misclassifications allowed. This means the hinge loss for all data points is zero.

### Soft Margin (for non-linearly separable data)

#### some data points may be misclassified, and a penalty is added to the objective function to account for these misclassifications. allows outlines 

### kernal trick  (for non-linear SVM)

#### kernel trick is a technique used in SVM to transform the input data into a higher-dimensional space, where it may become linearly separable. This allows SVM to find a hyperplane that can separate the classes e

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder,LabelEncoder,StandardScaler,MinMaxScaler

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


from sklearn.svm import SVC

In [19]:
from sklearn.metrics import accuracy_score

In [2]:
df= pd.read_csv("titanic_data_updated.csv")

df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
764,765,no,third,"Eklund, Mr. Hans Linus",male,16.0,0,0,347074,7.775,NaN,S
458,459,yes,second,"Toomey, Miss. Ellen",female,50.0,0,0,F.C.C. 13531,10.500,NaN,S
431,432,yes,third,"Thorneycroft, Mrs. Percival (Florence Kate White)",female,NaN,1,0,376564,16.100,NaN,S
887,888,yes,first,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.000,B42,S
184,185,yes,third,"Kink-Heilmann, Miss. Luise Gretchen",female,4.0,0,2,315153,22.025,NaN,S


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    object 
 2   Pclass       891 non-null    object 
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(3), object(7)
memory usage: 83.7+ KB


In [4]:
df['Family_Size'] = df['SibSp'] + df['Parch'] + 1
df['Cabin'] = df['Cabin'].fillna("Missing")

df['Deck'] = df['Cabin'].astype(str).str[0]
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
839,840,yes,first,"Marechal, Mr. Pierre",male,NaN,0,0,11774,29.7000,C47,C,1,C
36,37,yes,third,"Mamee, Mr. Hanna",male,NaN,0,0,2677,7.2292,Missing,C,1,M
247,248,yes,second,"Hamalainen, Mrs. William (Anna)",female,24.0,0,2,250649,14.5000,Missing,S,3,M
13,14,no,third,"Andersson, Mr. Anders Johan",male,39.0,1,5,347082,31.2750,Missing,S,7,M
711,712,no,first,"Klaber, Mr. Herman",male,NaN,0,0,113028,26.5500,C124,S,1,C


In [5]:
df['Deck'].value_counts()

Deck
M    687
C     59
B     47
D     33
E     32
A     15
F     13
G      4
T      1
Name: count, dtype: int64

In [6]:
X = df.drop(['Survived'], axis=1) 
y = df['Survived']



In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [9]:
X_train.sample(3)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
403,404,third,"Hakkarainen, Mr. Pekka Pietari",male,28.0,1,0,STON/O2. 3101279,15.8500,Missing,S,2,M
287,288,third,"Naidenoff, Mr. Penko",male,22.0,0,0,349206,7.8958,Missing,S,1,M
451,452,third,"Hagland, Mr. Ingvald Olai Olsen",male,NaN,1,0,65303,19.9667,Missing,S,2,M


In [10]:
y_train

692    yes
481     no
527     no
855    yes
801    yes
      ... 
359    yes
258    yes
736     no
462     no
507    yes
Name: Survived, Length: 712, dtype: object

In [11]:
#age
mean_age = X_train['Age'].mean()
std_age = X_train['Age'].std()

X_train['Z_score'] = (X_train['Age'] - mean_age) / std_age

musk = (abs(X_train['Z_score']) <= 3)

X_train = X_train[musk]
y_train = y_train[musk]

# fare

fare_Q1 = X_train['Fare'].quantile(0.25)
fare_Q3 = X_train['Fare'].quantile(0.75)

IQR = fare_Q3 - fare_Q1

minimum = max(0 , fare_Q1 - 1.5 * IQR)
maximum = fare_Q3 + 1.5 * IQR

X_train['Fare'] = X_train['Fare'].clip(minimum, maximum)

In [12]:
# pipeline

# numerical
p1 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='mean')),
        ('scaler',StandardScaler())
    ]
)

p2 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='median')),
        ('scaler',MinMaxScaler())
    ]
)

In [13]:
categories = [['third','second','first']]

In [14]:
 #pipeline 
#  categorical columns

p3 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OneHotEncoder(sparse_output=False,drop='first',handle_unknown='ignore'))
    ]
)

p4 = Pipeline(
    steps=[
        ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OrdinalEncoder(categories=categories)),
        ('scaler',MinMaxScaler())
    ]
)

In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        ('pipeline_1',p1,['Age']),
        ('pipeline_2',p2,['Fare','Family_Size']),
        ('pipeline_3',p3,['Embarked','Sex','Deck']),
        ('pipeline_4',p4,['Pclass'])
    ],
    remainder='drop'
)
preprocessor

,transformers,"[('pipeline_1', ...), ('pipeline_2', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'mean'
,fill_value,None


## 6. Target Variable Encoding

Our target column `Survived` contains 'yes' and 'no'. Since mathematical models require numbers, we use `LabelEncoder` to convert these to `1` and `0` respectively.

In [16]:
le = LabelEncoder()

le.fit(y_train)

y_train = le.transform(y_train)
y_test = le.transform(y_test)



## 7. Model Training and Hyperparameter Tuning

We don't just pick a model; we optimize it. We use **GridSearchCV** to test different settings (kernels and C values) for our Support Vector Classifier. This automatically finds the best configuration using Cross-Validation.

In [ ]:
SVC_model = Pipeline (
    steps =[
        ('prepocessor' , preprocessor),
        ('model' , SVC()) # 'model extends from the grid_param
    ]
)

## 8. Final Evaluation

Finally, we evaluate the model using three key metrics:
*   **Accuracy**: Overall correctness.
*   **Precision**: Out of those predicted to survive, how many actually did?
*   **Recall**: Out of all actual survivors, how many did the model find?

In [22]:

SVC_model.fit(X_train,y_train)

y_pred2 =SVC_model.predict(X_test)
print(accuracy_score(y_test,y_pred2))


0.8044692737430168


In [23]:
grid_param =[ {
    "model__kernel" : ['linear'], # transform features into higher dimension to find the hyperplane
    "model__C" :[0.01 , 0.1 , 1 , 10 , 50 , 100]   # smaller C means larger margin allows more mistake 
                                                     # larger C means smaller margin allows less mistake
},  # ^ models 
  {
      "model__kernel" : ['rbf'] , #
      "model__C" :[0.01 , 0.1 ,1,100],
      "model__gamma" :[0.01,0.1,5,10,'scale','auto'] # gamma controls How far the influence of one training point reaches.
  },  # 6*4 = 24 models
    {
        "model__kernel": ['poly'], # 
        "model__C" :[0.01 , 0.1 , 1 ,100],
        "model__degree" : [2,3] # degree means the degree of the polynomial kernel function ('poly'). It represents the flexibility of the decision boundary.
    }
 # 8 models  
 # total 6 + 24 + 8 = 38 models
]


In [25]:
from sklearn.model_selection import GridSearchCV

best_SVC_model = GridSearchCV(estimator = SVC_model , param_grid = grid_param , cv = 5)

In [26]:
best_SVC_model.fit(X_train,y_train)

/home/asus/.local/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/asus/.local/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/asus/.local/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/asus/.local/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/home/asus/.local/lib/python3.10/site-packages/sklearn/preprocessing/_encoders.py:24

,estimator,"Pipeline(step...del', SVC())])"
,param_grid,"[{'model__C': [0.01, 0.1, ...], 'model__kernel': ['linear']}, {'model__C': [0.01, 0.1, ...], 'model__gamma': [0.01, 0.1, ...], 'model__kernel': ['rbf']}, ...]"
,scoring,None
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('pipeline_1', ...), ('pipeline_2', ...), ...]"


In [27]:
# training accuracy
y_pred_train = best_SVC_model.predict(X_train)

print(accuracy_score(y_train,y_pred_train))

0.8429319371727748
